# Product Demand Prediction with Machine Learning

Finding the optimal discount price means knowing how demand reacts to price. This notebook
models **Units Sold** as a function of **Total Price** (the discounted selling price) and
**Base Price** (the original price), so we can ask *"if I sell at X instead of Y, how many
units should I expect?"*

**Dataset:** [demand.csv](https://raw.githubusercontent.com/amankharwal/Website-data/master/demand.csv)

| Column | Meaning |
| --- | --- |
| `ID` | Product ID |
| `Store ID` | Specific store ID |
| `Total Price` | Price at which the product was actually sold |
| `Base Price` | Initial (undiscounted) price of the product |
| `Units Sold` | Quantity demanded — **the target** |

## 1. Importing the libraries

In [1]:
import os

import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

## 2. Loading the dataset

In [2]:
DATA_PATH = "data/demand.csv"
DATA_URL = "https://raw.githubusercontent.com/amankharwal/Website-data/master/demand.csv"

data = pd.read_csv(DATA_PATH if os.path.exists(DATA_PATH) else DATA_URL)
data.head()

,ID,Store ID,Total Price,Base Price,Units Sold
0,1,8091,99.0375,111.8625,20
1,2,8091,99.0375,99.0375,28
2,3,8091,133.9500,133.9500,19
3,4,8091,133.9500,133.9500,44
4,5,8091,141.0750,141.0750,52


In [3]:
print("Rows, columns:", data.shape)
data.describe()

Rows, columns: (150150, 5)


,ID,Store ID,Total Price,Base Price,Units Sold
count,150150.000000,150150.000000,150149.000000,150150.000000,150150.000000
mean,106271.555504,9199.422511,206.626751,219.425927,51.674206
std,61386.037861,615.591445,103.308516,110.961712,60.207904
min,1.000000,8023.000000,41.325000,61.275000,1.000000
25%,53111.250000,8562.000000,130.387500,133.237500,20.000000
50%,106226.500000,9371.000000,198.075000,205.912500,35.000000
75%,159452.750000,9731.000000,233.700000,234.412500,62.000000
max,212644.000000,9984.000000,562.162500,562.162500,2876.000000


Two things already stand out from `describe()`: demand is heavily right-skewed (median 35 units,
maximum 2876), and prices span a wide range. Both matter later when we read the error metrics.

## 3. Visualizing the relationship between price and demand

In [4]:
# 150k points make an unreadable (and very heavy) chart, so we plot a random sample
plot_sample = data.sample(5000, random_state=42)

fig = px.scatter(
    plot_sample,
    x="Units Sold",
    y="Total Price",
    size="Units Sold",
    trendline="ols",
    title="Units sold vs. selling price",
)
fig.show()

The trendline slopes downward: higher price, lower quantity demanded. That is the classic
**law of demand**, and it confirms price carries real signal about sales volume. The relationship
is clearly not a straight line, though — the points fan out and cluster in price bands, which is
the first hint that a linear model will underfit.

In [5]:
fig = px.scatter(
    plot_sample,
    x="Base Price",
    y="Units Sold",
    color="Total Price",
    title="Demand across base price, coloured by actual selling price",
)
fig.show()

In [6]:
data.corr()["Units Sold"].sort_values(ascending=False)

Units Sold     1.000000
Store ID      -0.004372
ID            -0.010616
Base Price    -0.140032
Total Price   -0.235625
Name: Units Sold, dtype: float64

`Total Price` (-0.24) and `Base Price` (-0.14) both correlate negatively with demand, as expected.
`ID` and `Store ID` are essentially uncorrelated (|r| < 0.02) — they are identifiers, not
measurements, so we exclude them from the feature set.

## 4. Checking for and remove null values

In [7]:
data.isnull().sum()

ID             0
Store ID       0
Total Price    1
Base Price     0
Units Sold     0
dtype: int64

In [8]:
rows_before = len(data)
data = data.dropna()
print(f"Dropped {rows_before - len(data)} row(s) with missing values. Remaining: {len(data)}")

Dropped 1 row(s) with missing values. Remaining: 150149


Exactly one row has a missing `Total Price`. Dropping a single record out of 150,150 removes the
gap without any meaningful loss of information, so imputation is not worth the added complexity.

## 5. Splitting the data into training and test sets

In [9]:
x = data[["Total Price", "Base Price"]]
y = data["Units Sold"]

xtrain, xtest, ytrain, ytest = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print("Training set:", xtrain.shape)
print("Test set:    ", xtest.shape)

Training set: (120119, 2)
Test set:     (30030, 2)


An 80/20 split leaves ~120k rows for training and ~30k held-out rows for an honest evaluation.
`random_state=42` keeps the split reproducible across runs.

## 6. Training the model

In [10]:
model = DecisionTreeRegressor(random_state=42)
model.fit(xtrain, ytrain)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes 

### Why choose a decision tree?

The scatter plots showed a **non-linear, stepped** price/demand relationship: demand holds steady
across a price band and then drops sharply at certain thresholds. A decision tree splits the price
axis at exactly those thresholds instead of forcing one global slope through the data, and it needs
no feature scaling. The linear baseline below quantifies the difference.

## 7. Evaluating the model

In [11]:
train_score = model.score(xtrain, ytrain)
test_score = model.score(xtest, ytest)
mae = mean_absolute_error(ytest, model.predict(xtest))

print(f"R² on training data: {train_score:.4f}")
print(f"R² on test data:     {test_score:.4f}")
print(f"Mean absolute error: {mae:.2f} units")

R² on training data: 0.5939
R² on test data:     0.3765
Mean absolute error: 25.50 units


A fully grown tree memorises the training set (R² 0.59 train vs 0.38 test). That gap is overfitting,
so we compare depths to find where the tree generalises best.

In [12]:
results = []
for depth in [3, 5, 8, 10, 12, 15, 20, None]:
    candidate = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(xtrain, ytrain)
    results.append({
        "max_depth": depth,
        "train R²": round(candidate.score(xtrain, ytrain), 4),
        "test R²": round(candidate.score(xtest, ytest), 4),
    })

pd.DataFrame(results)

,max_depth,train R²,test R²
0,3.0,0.2093,0.2088
1,5.0,0.3266,0.3205
2,8.0,0.4178,0.4065
3,10.0,0.4623,0.4200
4,12.0,0.5067,0.4333
5,15.0,0.5598,0.4071
6,20.0,0.5919,0.3795
7,NaN,0.5939,0.3765


In [13]:
model = DecisionTreeRegressor(max_depth=12, random_state=42)
model.fit(xtrain, ytrain)

print(f"Final model R² (test):  {model.score(xtest, ytest):.4f}")
print(f"Final model R² (train): {model.score(xtrain, ytrain):.4f}")
print(f"Final model MAE:        {mean_absolute_error(ytest, model.predict(xtest)):.2f} units")

Final model R² (test):  0.4333
Final model R² (train): 0.5067
Final model MAE:        25.13 units


### Baseline comparison — linear regression

In [14]:
baseline = LinearRegression().fit(xtrain, ytrain)

print(f"Linear regression R² (test): {baseline.score(xtest, ytest):.4f}")
print(f"Linear regression MAE:       {mean_absolute_error(ytest, baseline.predict(xtest)):.2f} units")

Linear regression R² (test): 0.1488
Linear regression MAE:       32.49 units


The tuned tree roughly **triples** the explained variance of the linear baseline and cuts the average
error by about a quarter, which justifies the choice of model on this data rather than on theory alone.

## 8. Predicting demand for a given price

In [15]:
# features = [["Total Price", "Base Price"]]
features = pd.DataFrame([[133.00, 140.00]], columns=["Total Price", "Base Price"])

prediction = model.predict(features)
print(f"Selling at 133.00 against a base price of 140.00 -> {prediction[0]:.0f} units expected")

Selling at 133.00 against a base price of 140.00 -> 42 units expected


In [16]:
# How demand responds as we discount a product with a base price of 140
scenarios = pd.DataFrame(
    {"Total Price": [140.0, 133.0, 125.0, 115.0, 105.0], "Base Price": 140.0}
)
scenarios["Predicted Units Sold"] = model.predict(scenarios).round(0)
scenarios["Predicted Revenue"] = (scenarios["Total Price"] * scenarios["Predicted Units Sold"]).round(2)
scenarios

,Total Price,Base Price,Predicted Units Sold,Predicted Revenue
0,140.0,140.0,71.0,9940.0
1,133.0,140.0,42.0,5586.0
2,125.0,140.0,42.0,5250.0
3,115.0,140.0,49.0,5635.0
4,105.0,140.0,106.0,11130.0


This table is the practical payoff: for each candidate discount it gives an expected demand and the
revenue that follows, so a price can be chosen on evidence instead of intuition.

Note that the predicted curve is **not monotonic** — 133.00 is predicted to sell fewer units than the
undiscounted 140.00. The tree learns from whichever products historically sat in each price band, and
those bands contain different products, so a shallow discount can land in a band with weaker demand.
Treat the output as a band-level estimate, not a smooth demand curve for one specific item.

## Summary

* Demand falls as price rises, and the relationship is stepped rather than linear.
* A `DecisionTreeRegressor` limited to `max_depth=12` explains ~43% of the variance in units sold
  from price alone, against ~15% for linear regression.
* Remaining error is expected: price is only one of several demand drivers (promotions, seasonality,
  store location, stock levels), and none of the others are present in this dataset.

Full write-up, metrics and reasoning are in the [README](README.md).